# Part 2: Waste Material Classification with a CNN

We classify waste photos into 9 categories with transfer learning (a frozen MobileNetV2 plus our own layers), then tune it and study its mistakes.

Needs the folder `RealWaste` next to this notebook. Creates the file `waste_cnn.keras`, which Part 5 loads. Numbers in the Results come from my runs; training is not perfectly repeatable, so a rerun can differ by a point or two. Training runs on a CPU: allow about 15 minutes for the main run.

## Step 1: Loading Libraries

In [ ]:
import gc
import random
import hashlib
import pathlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import confusion_matrix, classification_report

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

np.random.seed(42)
random.seed(42)
tf.random.set_seed(42)

## Step 2: Load the Images

Same pipeline as Part 1: resize to 224 x 224, 80% for training and 20% held out.

In [ ]:
data_dir = pathlib.Path("RealWaste")
class_names = sorted([p.name for p in data_dir.glob("*") if p.is_dir()])
num_classes = len(class_names)

BATCH_SIZE = 32
IMG_HEIGHT = 224
IMG_WIDTH = 224

train_ds = keras.utils.image_dataset_from_directory(
    data_dir, validation_split=0.2, subset="training", seed=42,
    image_size=(IMG_HEIGHT, IMG_WIDTH), batch_size=BATCH_SIZE,
    label_mode="categorical", shuffle=True)

print("classes:", num_classes)
print(class_names)

## Step 3: Validation and Test Sets

The provided code split the hold-out with `take` and `skip` and leaked 129 images between validation and test (see Part 1). Here we read the 950 hold-out images once, keep each image with its own label, and split them by position with a stratified split.

In [ ]:
full_holdout = keras.utils.image_dataset_from_directory(
    data_dir, validation_split=0.2, subset="validation", seed=42,
    image_size=(IMG_HEIGHT, IMG_WIDTH), batch_size=BATCH_SIZE,
    label_mode="categorical", shuffle=True)

image_list, label_list = [], []
for images, labels in full_holdout:
    image_list.append(images.numpy())
    label_list.append(labels.numpy())
all_images = np.concatenate(image_list)
all_labels = np.concatenate(label_list)
label_numbers = all_labels.argmax(axis=1)

val_idx, test_idx = train_test_split(np.arange(len(label_numbers)), test_size=0.5,
                                     stratify=label_numbers, random_state=42)

validation_ds = tf.data.Dataset.from_tensor_slices((all_images[val_idx], all_labels[val_idx])).batch(BATCH_SIZE)
test_dataset = tf.data.Dataset.from_tensor_slices((all_images[test_idx], all_labels[test_idx])).batch(BATCH_SIZE)

print("validation per class:", np.bincount(label_numbers[val_idx]))
print("test per class:      ", np.bincount(label_numbers[test_idx]))

In [ ]:
def image_hashes(dataset):
    found = []
    for images, labels in dataset:
        for image in images.numpy():
            found.append(hashlib.md5(image.tobytes()).hexdigest())
    return found

val_hashes = image_hashes(validation_ds)
test_hashes = image_hashes(test_dataset)
print("images (validation, test):", len(val_hashes), len(test_hashes))
print("images in both:", len(set(val_hashes) & set(test_hashes)))

del all_images, all_labels, image_list, label_list, full_holdout
_ = gc.collect()

**Result:** 3802 training, 475 validation and 475 test images, with 0 images in both validation and test. Class counts are almost the same in validation and test; Textile Trash has only 37 and 38.

## Step 4: What the Model Receives

In [ ]:
print("TensorFlow:", tf.__version__)
print("GPUs:", tf.config.list_physical_devices("GPU"))

for images, labels in train_ds.take(1):
    print("image batch:", images.shape, images.dtype)
    print("label batch:", labels.shape)
    print("pixel range:", float(tf.reduce_min(images)), "to", float(tf.reduce_max(images)))

**Result:** No GPU, so training runs on the CPU. Batches are (32, 224, 224, 3) with one-hot labels (32, 9). Pixels run from 0 to 255, so the model rescales them to -1 to 1 itself (what MobileNetV2 was trained on).

## Step 5: Build the Model

MobileNetV2 with ImageNet weights is frozen, so it keeps what it already learned. Our layers on top: pooling, dropout, one 128-unit layer, and 9 outputs.

In [ ]:
base_model = keras.applications.MobileNetV2(input_shape=(IMG_HEIGHT, IMG_WIDTH, 3),
                                            include_top=False, weights="imagenet")
base_model.trainable = False

model = keras.Sequential([
    keras.Input(shape=(IMG_HEIGHT, IMG_WIDTH, 3)),
    layers.Rescaling(1 / 127.5, offset=-1),              # 0..255 -> -1..1
    base_model,
    layers.GlobalAveragePooling2D(name="gap"),
    layers.Dropout(0.3),
    layers.Dense(128, activation="relu"),
    layers.Dense(num_classes, activation="softmax"),
])

model.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-3),
              loss="categorical_crossentropy", metrics=["accuracy"])
model.summary()

**Result:** 2,423,113 parameters, of which only 165,129 are trained (the two Dense layers); the 2,257,984 in MobileNetV2 stay frozen. The base outputs 7 x 7 x 1280 and pooling turns that into 1280 numbers per image. Freezing first protects the pre-trained features from the random new layers and avoids overfitting on 3802 images.

## Step 6: Class Weights and Training

Errors on small classes count more. Weights come from the training labels only. Early stopping watches validation loss and keeps the best weights.

In [ ]:
train_labels = np.concatenate([labels.numpy().argmax(axis=1) for images, labels in train_ds])
weights = compute_class_weight("balanced", classes=np.arange(num_classes), y=train_labels)
class_weight = {i: w for i, w in enumerate(weights)}
for i, w in class_weight.items():
    print(class_names[i], round(float(w), 2))

early_stop = keras.callbacks.EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True)
history = model.fit(train_ds, validation_data=validation_ds, epochs=10,
                    class_weight=class_weight, callbacks=[early_stop])

**Result:** Weights run from Textile Trash 1.74 down to Plastic 0.57. Training stopped after epoch 5; the best validation loss (0.5550) was at epoch 2 (validation accuracy about 0.81), so the epoch-2 weights were kept. Training accuracy kept rising (0.76 to 0.89) while validation loss stopped improving: the start of overfitting, which early stopping reacts to.

## Step 7: Evaluate on the Test Set

Accuracy, the confusion matrix and per-class scores. Images and labels are read in the same loop so they stay paired.

In [ ]:
def evaluate_on_test(trained_model, title):
    true_list, pred_list = [], []
    for images, labels in test_dataset:
        true_list.append(labels.numpy().argmax(axis=1))
        pred_list.append(trained_model.predict(images, verbose=0).argmax(axis=1))
    y_true = np.concatenate(true_list)
    y_pred = np.concatenate(pred_list)
    print("test accuracy:", round(float((y_true == y_pred).mean()), 4))

    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(8, 7))
    plt.imshow(cm, cmap="Blues")
    plt.xticks(range(num_classes), class_names, rotation=45, ha="right")
    plt.yticks(range(num_classes), class_names)
    for i in range(num_classes):
        for j in range(num_classes):
            plt.text(j, i, cm[i, j], ha="center", va="center", color="white" if cm[i, j] > cm.max() / 2 else "black")
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.title(title)
    plt.tight_layout()
    plt.show()
    print(classification_report(y_true, y_pred, target_names=class_names))
    return y_true, y_pred

y_true, y_pred = evaluate_on_test(model, "Confusion matrix (test set, first model)")

**Result:** In an earlier run this first model scored about 0.79. Strongest: Vegetation (recall 0.97) and Food Organics (0.95). Weakest: Miscellaneous Trash (0.54), a catch-all class with no consistent look. Largest confusions: Plastic as Metal (11) and Metal as Plastic (10), Paper as Cardboard (7), Glass as Plastic (7). Part 1 predicted Paper/Cardboard; Plastic/Metal was a surprise.

## Step 8: Look at the Plastic and Metal Mistakes

Six images of each confusion, with the true and predicted class.

In [ ]:
def find_mistakes(trained_model, true_class, predicted_class, how_many=6):
    found = []
    for images, labels in test_dataset:
        predictions = trained_model.predict(images, verbose=0).argmax(axis=1)
        true_numbers = labels.numpy().argmax(axis=1)
        for image, t, p in zip(images.numpy(), true_numbers, predictions):
            if t == true_class and p == predicted_class:
                found.append(image)
    return found[:how_many]

plastic, metal = class_names.index("Plastic"), class_names.index("Metal")
plt.figure(figsize=(14, 5))
for row, (t, p) in enumerate([(plastic, metal), (metal, plastic)]):
    images = find_mistakes(model, t, p)
    for col, image in enumerate(images):
        plt.subplot(2, 6, row * 6 + col + 1)
        plt.imshow(image.astype("uint8"))
        plt.axis("off")
        plt.title(f"true {class_names[t]}\npred {class_names[p]}", fontsize=8)
plt.tight_layout()
plt.show()

**Result:** Most items carry printed labels or paint that hides the material; some plastic looks metallic (foil pouch, silver cap) and some metal looks like plastic (painted lid, mint tin); several items are crushed. A few are ambiguous even to a person. Based on 12 images.

## Step 9: Tuning the Custom Layers

MobileNetV2 is frozen, so we pass every image through it once, keep the 1280 numbers per image, and train different small heads on those numbers. Compared on validation only.

In [ ]:
extractor = keras.Sequential([keras.Input(shape=(IMG_HEIGHT, IMG_WIDTH, 3))] + model.layers[:3])

def get_features(dataset):
    features, labels_list = [], []
    for images, labels in dataset:
        features.append(extractor(images, training=False).numpy())
        labels_list.append(labels.numpy())
    return np.concatenate(features), np.concatenate(labels_list)

X_train, y_train = get_features(train_ds)
X_val, y_val = get_features(validation_ds)
print(X_train.shape, y_train.shape, X_val.shape, y_val.shape)

In [ ]:
def make_head(units=(128,), dropout=0.3, l2=0.0):
    regularizer = keras.regularizers.l2(l2) if l2 else None
    head = keras.Sequential([keras.Input(shape=(1280,)), layers.Dropout(dropout)])
    for i, u in enumerate(units):
        head.add(layers.Dense(u, activation="relu", kernel_regularizer=regularizer))
        if i < len(units) - 1:
            head.add(layers.Dropout(dropout))
    head.add(layers.Dense(num_classes, activation="softmax"))
    return head

def train_head(name, **settings):
    tf.random.set_seed(42)
    head = make_head(**settings)
    head.compile(optimizer=keras.optimizers.Adam(1e-3), loss="categorical_crossentropy", metrics=["accuracy"])
    stop = keras.callbacks.EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True)
    h = head.fit(X_train, y_train, validation_data=(X_val, y_val), epochs=30, batch_size=32,
                 class_weight=class_weight, callbacks=[stop], verbose=0)
    val_loss, val_acc = head.evaluate(X_val, y_val, verbose=0)
    row = {"run": name, "epochs": len(h.history["loss"]), "val_loss": round(val_loss, 4),
           "val_acc": round(val_acc, 4), "train_acc": round(h.history["accuracy"][-1], 4)}
    return head, row

In [ ]:
experiments = {
    "baseline: 128, dropout 0.3": dict(units=(128,), dropout=0.3),
    "more dropout: 128, 0.5": dict(units=(128,), dropout=0.5),
    "wider: 256, dropout 0.3": dict(units=(256,), dropout=0.3),
    "deeper: 256-128, dropout 0.3": dict(units=(256, 128), dropout=0.3),
    "L2 1e-3: 128, dropout 0.3": dict(units=(128,), dropout=0.3, l2=1e-3),
    "combo: 256, dropout 0.5": dict(units=(256,), dropout=0.5),
}

heads, rows = {}, []
for name, settings in experiments.items():
    heads[name], row = train_head(name, **settings)
    rows.append(row)
results = pd.DataFrame(rows)
print(results.to_string(index=False))

**Result:** In my run: baseline 0.6227 validation loss, more dropout 0.5831, wider 0.5801, deeper 0.6269, L2 0.7697, combo (256 units, dropout 0.5) 0.5433. Dropout 0.5 narrowed the train-validation accuracy gap (0.114 to 0.072); L2 did not help; a wider layer helped a little and a deeper one did not. The combo has the lowest validation loss and is chosen. Differences in accuracy are within noise (one validation image is worth 0.2 points), and each run is a single run.

## Step 10: Extra: scikit-learn Models on the Same Features

A second opinion on the same 1280 numbers: logistic regression, SVM and random forest, then cross-validation and a small grid search.

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, recall_score
from sklearn.model_selection import StratifiedKFold, cross_val_score, GridSearchCV

train_numbers = y_train.argmax(axis=1)
val_numbers = y_val.argmax(axis=1)

sk_models = {
    "Logistic regression": make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, class_weight="balanced")),
    "SVM (rbf)": make_pipeline(StandardScaler(), SVC(kernel="rbf", class_weight="balanced")),
    "Random forest": RandomForestClassifier(n_estimators=300, class_weight="balanced", random_state=42),
}
rows = []
for name, sk_model in sk_models.items():
    sk_model.fit(X_train, train_numbers)
    pred = sk_model.predict(X_val)
    rows.append({"model": name, "accuracy": round(accuracy_score(val_numbers, pred), 4),
                 "macro F1": round(f1_score(val_numbers, pred, average="macro"), 4)})

best_name = results.loc[results["val_loss"].idxmin(), "run"]
pred = heads[best_name].predict(X_val, verbose=0).argmax(axis=1)
rows.append({"model": "Keras head: " + best_name, "accuracy": round(accuracy_score(val_numbers, pred), 4),
             "macro F1": round(f1_score(val_numbers, pred, average="macro"), 4)})
print(pd.DataFrame(rows).to_string(index=False))

In [ ]:
folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
logistic = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, class_weight="balanced"))
scores = cross_val_score(logistic, X_train, train_numbers, cv=folds, scoring="accuracy")
print("5-fold accuracy:", scores.round(4), "mean:", round(scores.mean(), 4), "spread:", round(scores.std(), 4))

grid = GridSearchCV(make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, class_weight="balanced")),
                    param_grid={"logisticregression__C": [0.001, 0.01, 0.1, 1]}, cv=3, scoring="f1_macro")
grid.fit(X_train, train_numbers)
print("best C:", grid.best_params_, "| validation accuracy:", round(accuracy_score(val_numbers, grid.predict(X_val)), 4))

In [ ]:
# do class weights help? train the chosen head with and without them
def head_predictions(use_weights):
    tf.random.set_seed(42)
    head = make_head(**experiments[best_name])
    head.compile(optimizer=keras.optimizers.Adam(1e-3), loss="categorical_crossentropy", metrics=["accuracy"])
    stop = keras.callbacks.EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True)
    head.fit(X_train, y_train, validation_data=(X_val, y_val), epochs=30, batch_size=32,
             class_weight=class_weight if use_weights else None, callbacks=[stop], verbose=0)
    return head.predict(X_val, verbose=0).argmax(axis=1)

weighted = head_predictions(True)
unweighted = head_predictions(False)
table = pd.DataFrame({"with weights": recall_score(val_numbers, weighted, average=None).round(3),
                      "without weights": recall_score(val_numbers, unweighted, average=None).round(3)},
                     index=class_names)
print(table)

In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components=2, random_state=42).fit(X_train)       # fitted on training data only
points = pca.transform(X_val)
print("variance kept by 2 components:", pca.explained_variance_ratio_.round(3))

plt.figure(figsize=(8, 6))
for k, name in enumerate(class_names):
    plt.scatter(points[val_numbers == k, 0], points[val_numbers == k, 1], s=14, alpha=0.6, label=name)
plt.xlabel("PCA 1")
plt.ylabel("PCA 2")
plt.legend(fontsize=7)
plt.title("Validation features in 2 dimensions")
plt.show()

**Result:** Run this step and note which model has the best validation accuracy, whether cross-validation agrees with the single validation score, whether Plastic recall changes without class weights, and which classes overlap in the plot. The final CNN stays the Keras model, because Part 5 needs one model that takes an image.

## Step 11: Final Model and Test Evaluation

The head with the lowest validation loss is joined to the frozen base and scored once on the test set. The model is saved for Part 5.

In [ ]:
best_name = results.loc[results["val_loss"].idxmin(), "run"]
print("chosen by lowest validation loss:", best_name)

final_model = keras.Sequential([extractor, heads[best_name]])
final_model.compile(loss="categorical_crossentropy", metrics=["accuracy"])

y_true, y_pred = evaluate_on_test(final_model, "Confusion matrix (test set, tuned model)")
final_model.save("waste_cnn.keras")

**Result:** Test accuracy 0.8168 (macro F1 0.82) on 475 images; the same architecture scored 0.84 in an earlier run, so a single run varies by about 2 points. Recall: Vegetation 0.97, Food Organics and Paper 0.93, Glass 0.90, Textile Trash 0.84, Cardboard and Metal 0.78, Plastic 0.76, Miscellaneous Trash 0.57 (weakest). Plastic and Metal are often called Glass, so Glass has the lowest precision (0.69).

## Summary

- **Model:** frozen MobileNetV2 plus dropout 0.5 and a 256-unit layer, trained with class weights and early stopping. Test accuracy 0.8168.
- **Hardest classes:** Miscellaneous Trash and Plastic. The largest confusion is Plastic and Metal (printed or painted surfaces, metallic-looking plastic, crushed items).
- **Limitations:** single runs, small validation and test sets (Textile Trash has about 38 images), CPU training so no large experiments, only 12 mistake images viewed, and only one base model tried.